# 03 · Join Sofascore + Capology — Spain La Liga 25/26 (snapshot 20260428)

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2025/26 de la La Liga española**.

⚠️ **Nota sobre el snapshot:** la temporada 25/26 está aún en curso. Se trabaja con
una foto fija de Sofascore (`df_spain_2526_snapshot_20260428.csv`). Este notebook
deberá reejecutarse con los datos definitivos cuando finalice la liga, generando
entonces el master sin sufijo de fecha (`master_spain_2526.csv`).

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_spain_2526_snapshot_20260428.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_spain_2526.csv').copy()

print(f'Sofascore (snapshot):  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:              {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore (snapshot):  578 jugadores | 117 columnas
Capology:              525 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   deportivo alaves
   fc barcelona
   girona fc
   levante ud

En Capology pero no en Sofascore:
   alaves
   barcelona
   girona
   levante


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'alaves':'deportivo alaves',
            'barcelona':'fc barcelona',
            'girona':'girona fc',
            'levante':'levante ud'

}

# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 463/578 (80.1%)
Sin emparejar: 115


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          8
Revisión media    (0.75 ≤ score < 0.90):   12
Revisión estricta (0.50 ≤ score < 0.75):   52
Revisión muy est. (score < 0.50):           43


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
23,Yáser Asprilla,Girona FC,yaser asprilla,1.000
14,Alexander Sørloth,Atlético Madrid,alexander sorloth,0.970
8,Viktor Tsygankov,Girona FC,viktor tsyhankov,0.938
20,Dani Carvajal,Real Madrid,daniel carvajal,0.929
60,Manuel Sánchez,Levante UD,manu sanchez,0.923
32,Javier Guerra,Valencia,javi guerra,0.917
18,Javier Rueda,Celta Vigo,javi rueda,0.909
55,Dani Raba,Valencia,daniel raba,0.900


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
78,Tai Abed,Levante UD,tay abed,0.875
15,Josep Chavarría,Rayo Vallecano,pep chavarria,0.857
33,Alejandro Rego Mora,Athletic Club,alejandro rego,0.848
81,Justin-Noel Kalumba,Mallorca,justin kalumba,0.848
1,Abdessamad Ezzalzouli,Real Betis,abde ezzalzouli,0.833
19,Alexander Freeman,Villarreal,alex freeman,0.828
72,Damián Rodríguez,Celta Vigo,javi rodriguez,0.800
36,Etta Eyong,Levante UD,karl etta eyong,0.800
10,Jose Maria Gimenez,Atlético Madrid,jose gimenez,0.800
17,Orri Steinn Óskarsson,Real Sociedad,orri oskarsson,0.800


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────

EXCLUDE_FROM_FUZZY = ['damian rodriguez'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 11 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
7,Pathé Ismaël Ciss,Rayo Vallecano,pathe ciss,0.741
22,Raúl García de Haro,Osasuna,raul garcia,0.733
80,Juan Arango,Girona FC,juan carlos,0.727
37,Urko González,Espanyol,urko gonzalez de zarate,0.722
66,Hugo González,Celta Vigo,hugo alvarez,0.720
0,Pablo Cuñat,Levante UD,pablo campos,0.696
90,Jofre Torrents,FC Barcelona,ferran torres,0.667
45,Álex Sola,Getafe,alex sancris,0.667
9,Carlos Vicente,Deportivo Alavés,carlos benavidez,0.667
34,A. J. Morales,Levante UD,jose luis morales,0.643


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['pathe ismael ciss',
                    'raul garcia de haro',
                    'urko gonzalez',
                    'pablo cunat',
                    'a j morales',
                    'abdon',
                    'johnny cardoso',
                    'pablo gavi',
                    'djene dakonam',
                    'alemao'
                    

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 10


### 7.4 Revisión muy estricta (score < 0.50)

Candidatos con muy baja similitud. Por defecto ninguno se acepta.
Añadir a `ACCEPT_VERY_LOW_FUZZY` los que se confirmen manualmente.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
52,Manuel Ángel Morán,Real Madrid,aurelien tchouameni,0.486
16,Manor Solomon,Villarreal,santiago mourino,0.483
96,Jorge Cabello,Levante UD,roger brugue,0.480
91,Moussa Diarra,Deportivo Alavés,mariano diaz,0.480
54,Marc Domenech,Mallorca,mateo joseph,0.480
97,Pablo Agudín,Real Oviedo,alberto reina,0.480
6,Adrià Altimira,Villarreal,arnau tenas,0.480
29,Rubén Iranzo,Valencia,lucas beltran,0.480
27,Job Ochieng,Real Sociedad,jon martin,0.476
48,Xavi Espart,FC Barcelona,inaki pena,0.476


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 491/578 (84.9%)
Sin salario:     87


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 87


,player,team,minutesPlayed,appearances,goals,assists
0,Eder Garcia,Athletic Club,36,2,0,0
1,Asier Hierro,Athletic Club,33,1,0,0
2,Conor Gallagher,Atlético Madrid,665,19,2,0
3,Giacomo Raspadori,Atlético Madrid,250,12,0,1
4,Julio Díaz,Atlético Madrid,221,3,0,1
5,Javier Boñar,Atlético Madrid,180,2,1,0
6,Dani Martinez,Atlético Madrid,90,1,0,0
7,Rayane Belaid,Atlético Madrid,71,1,0,0
8,Javi Morcillo,Atlético Madrid,48,2,0,0
9,Jano Monserrate,Atlético Madrid,28,2,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Athletic Club  —  SF sin salario:


,player,minutesPlayed
0,Asier Hierro,33
1,Eder Garcia,36


  CG plantilla completa:


,player,player_norm
0,Adama Boiro,adama boiro
1,Aitor Paredes,aitor paredes
2,Alejandro Rego,alejandro rego
3,Álex Berenguer,alex berenguer
4,Álex Padilla,alex padilla
5,Andoni Gorosabel,andoni gorosabel
6,Aymeric Laporte,aymeric laporte
7,Beñat Prados,benat prados
8,Dani Vivian,dani vivian
9,Gorka Guruzeta,gorka guruzeta



  Atlético Madrid  —  SF sin salario:


...

,player,player_norm
0,André Almeida,andre almeida
1,Arnaut Danjuma,arnaut danjuma
2,Baptiste Santamaria,baptiste santamaria
3,César Tárrega,cesar tarrega
4,Cristian Rivero,cristian rivero
5,Daniel Raba,daniel raba
6,Diego López,diego lopez
7,Dimitri Foulquier,dimitri foulquier
8,Eray Cömert,eray comert
9,Filip Ugrinic,filip ugrinic



  Villarreal  —  SF sin salario:


,player,minutesPlayed
0,Adrià Altimira,8
1,Hugo Lopez,55
2,Manor Solomon,153
3,Yéremy Pino,160


  CG plantilla completa:


,player,player_norm
0,Alberto Moleiro,alberto moleiro
1,Alex Freeman,alex freeman
2,Alfon González,alfon gonzalez
3,Alfonso Pedraza,alfonso pedraza
4,Arnau Tenas,arnau tenas
5,Ayoze Pérez,ayoze perez
6,Carlos Macià,carlos macia
7,Dani Parejo,dani parejo
8,Diego Conde,diego conde
9,Georges Mikautadze,georges mikautadze


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('pacha',       'rayo vallecano'): ('alfonso espino',  'rayo vallecano'),
    ('abdel rahim', 'real oviedo')   : ('rahim alhassane', 'real oviedo'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')

Matches manuales definidos: 2


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: pacha (rayo vallecano) → alfonso espino (rayo vallecano)
✅ Match manual aplicado: abdel rahim (real oviedo) → rahim alhassane (real oviedo)

Tras matches manuales: 493/578 (85.3%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

⚠️ Nombre con sufijo `_snapshot_20260428` para diferenciar del master definitivo
que se generará al cierre de la temporada (`master_spain_2526.csv`).

In [21]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_spain_2526_snapshot_20260428.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_spain_2526_snapshot_20260428.csv
   Jugadores totales:  578
   Con salario:        493
   Sin salario (NaN):  85
   Columnas:           122
